# Jet turbulence (Modane): data and MGD model
Data: velocity record of the Modane wind tunnel. Model: run `jets_full`, 5 seeds (500–504)
(wavelet marginals on the Morlet bank, Q = 1; scattering coefficients of order 1 to 4; $L^6$ norm).
$\Theta$ = $\Theta_{\rm reg}$ at the last time step of the transport ($t \approx 1$), $\lambda = 10^{-2}$.

Run on Jean Zay from `jets/`. Uses `figures/logp_jets/logp_jets.npz` (`python jets/logp_jets.py`). Figures: `figures/talk/`.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
ROOT = Path.cwd()                                   # jets/
for p in ('codes', 'codes/filters', 'codes/potentials', 'data', 'turbulence', 'jets'):
    sys.path.insert(0, str(ROOT.parent / p))
import json
import numpy as np
import pandas as pd
import torch
from IPython.display import display
import talk_figures as tf
import run_comparison as rc
from logp_family_contrib import load_potentials
tf.set_output(ROOT / 'figures' / 'talk')          # every figure is also saved here as PNG

In [ ]:
import jets_compare as jc
import logp_jets as lj
from jet_data import load_modane_raw
LABEL, SEEDS = 'jets_full', [500, 501, 502, 503, 504]
configs = jc.find_seed_configs(ROOT, LABEL, SEEDS)
runs = {s: rc.load_config(c, ROOT, label=LABEL, extras=False) for s, c in configs.items()}
cfg = runs[SEEDS[0]]['cfg']
x1 = jc.reference_data(cfg).numpy()                                   # training data (n, T), standardized
xt = np.concatenate([r['xt'].numpy() for r in runs.values()])         # generated signals, 5 seeds
L = lj.load(ROOT / 'figures' / 'logp_jets' / 'logp_jets.npz')['jets_full']
assert sorted(int(s) for s in L['seeds']) == SEEDS and L['C']['train'].shape[-1] == len(x1)
# Theta(t = 1) of seed 500 at the selected lambda (the file logp_jets.py used)
from logp_seed_check import XSEED
xsel = torch.load(XSEED / 'xseed_jets_full.pt', map_location='cpu', weights_only=False)
sol = torch.load(xsel['theta_files']['end0.99'][configs[SEEDS[0]]], map_location='cpu', weights_only=False)
assert sol['config'] == configs[SEEDS[0]] and float(sol['lam']) == L['lam'], (sol['config'], sol['lam'])
theta = sol['Theta_reg'][-1].double().numpy()
print('data', x1.shape, '| generated', xt.shape, '| lambda', L['lam'])

## Contribution of each potential family to log p
For each data signal, $C_F(x) = \Theta_F \cdot \phi_F(x)$, and $\log p(x) = \sum_F C_F(x)$ + constant.
Top: mean over signals (its zero is arbitrary: a constant added to $\phi_F$ changes it, not $p$).
Middle: standard deviation across signals. Bottom: the same divided by the standard deviation of $\log p$.
Error bars: standard deviation over the 5 seeds.

In [ ]:
C = L['C']['train']                                   # (seeds, families + total, signals)
tf.plot_energy_bars({f: C[:, i] for i, f in enumerate(L['families'])}, 'logp_per_family')

## Distribution of log p over the data
$\log p(x) = \Theta \cdot \phi(x)$ up to a constant, for the training signals; each seed relative to its own
median, the 5 seeds pooled. The contribution of the third- and fourth-order scattering terms is in the figure above.

In [ ]:
tf.plot_logp_histogram(np.concatenate([c[-1] - np.median(c[-1]) for c in C]), 'logp_data')

## Wavelet coefficients of the data, per scale
Real part of the Morlet wavelet coefficients, one panel per scale $j$, divided by their standard deviation.
Dashed: standard Gaussian.

In [ ]:
tf.plot_wavelet_histograms(x1, int(cfg['J']), 'data_wavelet_histograms')

## Signals, zoomed
Top: 6 stretches of 500 samples of the raw record (standardized), chosen at random.
Bottom: 6 generated signals (256 samples; one sample = 2 raw samples).

In [ ]:
u = load_modane_raw()
u = (u - u.mean()) / u.std()
rng = np.random.default_rng(0)
starts = rng.choice(len(u) - 500, 6, replace=False)
tf.plot_signals(np.stack([u[s:s + 500] for s in starts]), 'raw_record_zoom', xlabel='time (raw samples)')
tf.plot_signals(xt[rng.choice(len(xt), 6, replace=False)], 'generated_signals', color=tf.MODEL)

## Number of statistics in the model
Per potential family, as used in the run (seed 500; after removing duplicate filters and empty windows).

In [ ]:
from logp_family_contrib import phi_matrix
exp = rc._find_exp_dir(ROOT, configs[SEEDS[0]])
pots = load_potentials(exp, cfg, 'cpu')
xs = torch.as_tensor(x1[:200]).reshape(200, 1, -1).float()
# check: log p of 200 data signals recomputed here = the saved values (same Theta, potentials and sign)
lp_check = phi_matrix(pots, xs) @ theta
assert np.allclose(lp_check, C[[int(s) for s in L['seeds']].index(SEEDS[0]), -1][:200], rtol=1e-6, atol=1e-6 * np.abs(lp_check).max()), 'saved log p differs'
print('log p of 200 data signals recomputed: matches the saved values')
n_stats = tf.count_statistics(pots, xs, theta, Q=cfg['Q'])
n_stats.to_csv(ROOT / 'figures' / 'talk' / 'number_of_statistics.csv')
display(n_stats)